# Attention Residual Adapter Analysis

This notebook demonstrates how to analyze which earlier layers are most frequently attended to and examine attention patterns across different tasks/datasets.

## Goals:
1. Track which earlier layers each depth position attends to
2. Analyze task-specific attention patterns
3. Compare gate activations across layers and datasets
4. Visualize attention flow through the model

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import importlib
import sys

# Import our analysis modules with force reload
# sys.path.insert(0, '/home/students/wli/UniHeidelberg/semster3/IsAttAll/PEFT-style-AttnRes-adapter')

# # Force reload of analysis modules to pick up latest changes
# if 'src.attnres_analyzer' in sys.modules:
#     del sys.modules['src.attnres_analyzer']
# if 'src.attnres_visualizer' in sys.modules:
#     del sys.modules['src.attnres_visualizer']

from attnres_analyzer import AttnResAnalyzer, AttnResHook
from attnres_visualizer import AttnResVisualizer

print("Imports successful!")

Imports successful!


## Step 1: Load the Model

Load your finetuned wrapped model with Attention Residual adapters.

In [2]:
# Update these paths if necessary to actual model locations
model_path = "/home/students/wli/UniHeidelberg/semster2/final_projects/models/Qwen3-0.6B-Base"
tokenizer_path = model_path

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)

print(f"Base model loaded: {base_model.__class__.__name__}")
print(f"Number of layers: {len(base_model.model.layers)}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Base model loaded: Qwen3ForCausalLM
Number of layers: 28


In [3]:
# Wrap the model with AttnRes adapters
from sample import load_qwen3_attnres_model

# Reload sample module to get latest implementation
import sample
sample = importlib.reload(sample)
from sample import load_qwen3_attnres_model

# Important: Store the lookback value used in the model
lookback_window = 8  # This must match the lookback value used in load_qwen3_attnres_model
attnres_model = load_qwen3_attnres_model(base_model, lookback=lookback_window, gate_init=0.1)

# Optional: Load checkpoint if you have one
# checkpoint_path = "./checkpoints/attnres_checkpoint.pt"
# checkpoint = torch.load(checkpoint_path, map_location='cpu')
# attnres_model.load_state_dict(checkpoint, strict=False)

print(f"AttnRes model created with {len(attnres_model.adapters)} adapters")
print(f"Lookback window: {lookback_window}")

AttnRes model created with 28 adapters
Lookback window: 8


## Step 2: Prepare Datasets

Create datasets for different tasks to analyze cross-task attention patterns.

In [4]:
# Define different task/datasets
reasoning_texts = [
    "Solve this problem: What is 2+2? Let me think step by step.",
    "Here's a logical puzzle: If A is greater than B and B is greater than C, what can we conclude?",
    "Mathematical reasoning: Prove that the square of an odd number is always odd.",
    "If there are 5 apples and 3 oranges, how many pieces of fruit are there total?",
    "Consider this: All dogs are animals. Fido is a dog. Therefore, Fido is an animal. This is what type of reasoning?",
] * 3

summarization_texts = [
    "Please summarize: The quick brown fox jumps over the lazy dog in a field of wildflowers during spring.",
    "Write a short summary of: Climate change is affecting ecosystems worldwide through rising temperatures and changing precipitation patterns.",
    "Condense: Machine learning is a subset of artificial intelligence that focuses on learning from data and improving performance without explicit programming.",
    "Summarize: The Industrial Revolution transformed manufacturing through mechanization, steam power, and factory systems.",
    "Abstract: Python is a high-level programming language known for its simplicity, readability, and versatile applications in data science, web development, and automation.",
] * 3

qa_texts = [
    "What is the capital of France?",
    "How does photosynthesis work in plants?",
    "Explain quantum computing in simple terms.",
    "What are the main causes of ocean acidification?",
    "Describe the process of cellular respiration.",
] * 3

print(f"Reasoning dataset: {len(reasoning_texts)} samples")
print(f"Summarization dataset: {len(summarization_texts)} samples")
print(f"QA dataset: {len(qa_texts)} samples")

Reasoning dataset: 15 samples
Summarization dataset: 15 samples
QA dataset: 15 samples


In [5]:
# Create a simple dataset class
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        tokens = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }

# Create dataloaders
reasoning_loader = DataLoader(TextDataset(reasoning_texts, tokenizer), batch_size=4, shuffle=False)
summarization_loader = DataLoader(TextDataset(summarization_texts, tokenizer), batch_size=4, shuffle=False)
qa_loader = DataLoader(TextDataset(qa_texts, tokenizer), batch_size=4, shuffle=False)

print("Dataloaders created!")

Dataloaders created!


## Step 3: Analyze Each Dataset

Run inference on each dataset and collect attention residual statistics.

In [6]:
def analyze_dataset(model, dataloader, dataset_id, dataset_name, lookback=None, device='cuda'):
    """
    Run inference on a dataset and collect adapter usage statistics.
    
    Args:
        model: The wrapped model with adapters
        dataloader: DataLoader for the dataset
        dataset_id: Unique identifier for the dataset
        dataset_name: Human-readable name for the dataset
        lookback: Lookback window size (e.g., 8 means each adapter can attend to up to 8 earlier layers)
        device: Device to run on ('cuda' or 'cpu')
    """
    num_layers = len(model.model.layers)
    analyzer = AttnResAnalyzer(num_layers=num_layers, lookback=lookback)
    analyzer.set_dataset_name(dataset_id, dataset_name)
    
    model.eval()
    
    print(f"\nAnalyzing {dataset_name}...")
    with torch.no_grad(), AttnResHook(model, analyzer, dataset_id=dataset_id):
        for batch_idx, batch in enumerate(dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch.get('attention_mask', None)
            if attention_mask is not None:
                attention_mask = attention_mask.to(device)
            
            # Forward pass (hooks automatically capture statistics)
            _ = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_depth_weights=False,
            )
            
            if (batch_idx + 1) % 3 == 0:
                print(f"  Processed {(batch_idx + 1) * 4} samples")
    
    print(f"✓ {dataset_name} analysis complete")
    return analyzer

# Analyze each dataset with lookback constraint
reasoning_analyzer = analyze_dataset(attnres_model, reasoning_loader, "reasoning", "Reasoning", lookback=lookback_window, device=next(attnres_model.parameters()).device)
summarization_analyzer = analyze_dataset(attnres_model, summarization_loader, "summarization", "Summarization", lookback=lookback_window, device=next(attnres_model.parameters()).device)
qa_analyzer = analyze_dataset(attnres_model, qa_loader, "qa", "Q&A", lookback=lookback_window, device=next(attnres_model.parameters()).device)


Analyzing Reasoning...
  Processed 12 samples
✓ Reasoning analysis complete

Analyzing Summarization...
  Processed 12 samples
✓ Summarization analysis complete

Analyzing Q&A...
  Processed 12 samples
✓ Q&A analysis complete


## Step 4: Combine Results and Compute Statistics

In [7]:
# Combine all analyzers
num_layers = len(attnres_model.model.layers)
combined_analyzer = AttnResAnalyzer(num_layers=num_layers, num_datasets=3, lookback=lookback_window)

# Merge individual datasets into combined analyzer
for analyzer, dataset_id in [
    (reasoning_analyzer, "reasoning"),
    (summarization_analyzer, "summarization"),
    (qa_analyzer, "qa"),
]:
    combined_analyzer.dataset_names[dataset_id] = analyzer.dataset_names.get(dataset_id, dataset_id)
    combined_analyzer.dataset_sample_counts[dataset_id] = analyzer.dataset_sample_counts[dataset_id]
    
    for layer_idx in range(num_layers):
        for ds_id, alphas in analyzer.layer_source_attention[layer_idx].items():
            combined_analyzer.layer_source_attention[layer_idx][ds_id].extend(alphas)
        for ds_id, gates in analyzer.layer_gate_values[layer_idx].items():
            combined_analyzer.layer_gate_values[layer_idx][ds_id].extend(gates)
        for ds_id, norms in analyzer.layer_adapter_output_norm[layer_idx].items():
            combined_analyzer.layer_adapter_output_norm[layer_idx][ds_id].extend(norms)

print("Analyzers combined with lookback constraint applied!")

Analyzers combined with lookback constraint applied!


In [8]:
# Compute aggregated statistics
stats = combined_analyzer.compute_aggregated_stats()

# Print summary
combined_analyzer.print_summary()


ATTENTION RESIDUAL ADAPTER ANALYSIS SUMMARY

Dataset Overview:
--------------------------------------------------------------------------------
  Reasoning: 420 samples
  Summarization: 420 samples
  Q&A: 420 samples

Per-Layer Depth Attention (which earlier layers are attended to):
--------------------------------------------------------------------------------
  Layer 0:
    Most attended sources: [('Embedding', 1.0)]
    Attention distribution: ['1.000']...
  Layer 1:
    Most attended sources: [('Embedding', 0.521410346031189), ('Layer 0', 0.47858962416648865)]
    Attention distribution: ['0.521', '0.479']...
  Layer 2:
    Most attended sources: [('Embedding', 0.3470403850078583), ('Layer 0', 0.33219411969184875), ('Layer 1', 0.3207654058933258)]
    Attention distribution: ['0.347', '0.332', '0.321']...
  Layer 3:
    Most attended sources: [('Layer 0', 0.28821879625320435), ('Layer 1', 0.2473529428243637), ('Embedding', 0.24671433866024017)]
    Attention distribution: ['0.247

## Step 5: Analyze Most Attended Layers

Find which earlier layers are most frequently attended to.

## Important: Lookback Window Constraint

The Attention Residual adapter respects a **lookback window** constraint. With `lookback=8`:
- Layer 0: Can attend to NOTHING (first layer)
- Layer 1: Can attend to layer 0 (1 layer back)
- Layer 8: Can attend to layers 0-7 (8 layers back)
- Layer 9: Can attend to layers 1-8 (8 most recent layers)
- Layer 16: Can attend to layers 8-15 (8 most recent layers)
- Layer 23: Can attend to layers 15-22 (8 most recent layers)

Our analysis now respects this constraint - only layers within the valid windows are shown.

In [9]:
# Verify lookback constraint is respected
print("="*80)
print("VALIDATION: Checking Lookback Constraint (lookback={})\n".format(lookback_window))

for check_layer_idx in range(num_layers):
    if check_layer_idx < num_layers:
        top_layers = combined_analyzer.get_most_attended_layers(check_layer_idx, top_k=5)
        
        # Calculate valid window
        valid_start = max(0, check_layer_idx - lookback_window)
        valid_end = check_layer_idx
        
        print(f"Layer {check_layer_idx}: Valid window = [{valid_start}, {valid_end-1}]")
        if top_layers:
            print(f"  Top attended layers: {[src_idx for src_idx, _ in top_layers]}")
            
            # Check all are within bounds
            all_valid = all(valid_start <= src_idx < valid_end for src_idx, _ in top_layers)
            status = "✓ VALID" if all_valid else "✗ INVALID"
            print(f"  {status}")
        else:
            print(f"  No attention data")
        print()

VALIDATION: Checking Lookback Constraint (lookback=8)

Layer 0: Valid window = [0, -1]
  Top attended layers: [-1]
  ✗ INVALID

Layer 1: Valid window = [0, 0]
  Top attended layers: [-1, 0]
  ✗ INVALID

Layer 2: Valid window = [0, 1]
  Top attended layers: [-1, 0, 1]
  ✗ INVALID

Layer 3: Valid window = [0, 2]
  Top attended layers: [0, 1, -1, 2]
  ✗ INVALID

Layer 4: Valid window = [0, 3]
  Top attended layers: [2, 1, 0, 3, -1]
  ✗ INVALID

Layer 5: Valid window = [0, 4]
  Top attended layers: [3, 0, -1, 4, 2]
  ✗ INVALID

Layer 6: Valid window = [0, 5]
  Top attended layers: [2, 1, 5, 0, 3]
  ✓ VALID

Layer 7: Valid window = [0, 6]
  Top attended layers: [-1, 1, 2, 3, 0]
  ✗ INVALID

Layer 8: Valid window = [0, 7]
  Top attended layers: [2, 5, 7, 4, 3]
  ✓ VALID

Layer 9: Valid window = [1, 8]
  Top attended layers: [7, 1, 6, 4, 2]
  ✓ VALID

Layer 10: Valid window = [2, 9]
  Top attended layers: [9, 6, 7, 8, 5]
  ✓ VALID

Layer 11: Valid window = [3, 10]
  Top attended layers: [8, 1

In [10]:
# Verify the lookback constraint is now properly applied to aggregated stats
print("\n" + "="*80)
print("VERIFICATION: Lookback Constraint Applied to Aggregated Statistics")
print("="*80)

layer_stats = stats['layer_depth_stats']

test_layers = [0, 5, 8, 9, 14, 16, 18, 20, 23]

for layer_idx in test_layers:
    if layer_idx < num_layers and layer_idx in layer_stats:
        layer_stat = layer_stats[layer_idx]
        
        # Calculate valid window
        valid_start = max(0, layer_idx - lookback_window)
        valid_end = layer_idx
        
        # Get the ranking from aggregated stats
        if layer_stat.get('aggregated'):
            ranking = layer_stat['aggregated'].get('layer_ranking', [])[:5]
            
            print(f"\nLayer {layer_idx}: Valid window = [{valid_start:2d}, {valid_end-1:2d}]")
            
            if ranking:
                attended = [f"L{src}({w:.3f})" for src, w in ranking]
                print(f"           Top 5 attended: {', '.join(attended)}")
                
                # Verify all are within bounds
                all_valid = all(valid_start <= src <= valid_end-1 for src, _ in ranking)
                status = "✓ VALID" if all_valid else "✗ INVALID"
                print(f"           {status}")
            else:
                print(f"           No ranking data")


VERIFICATION: Lookback Constraint Applied to Aggregated Statistics

Layer 0: Valid window = [ 0, -1]
           Top 5 attended: L-1(1.000)
           ✗ INVALID

Layer 5: Valid window = [ 0,  4]
           Top 5 attended: L3(0.195), L0(0.185), L-1(0.174), L4(0.164), L2(0.144)
           ✗ INVALID

Layer 8: Valid window = [ 0,  7]
           Top 5 attended: L2(0.136), L5(0.133), L7(0.129), L4(0.127), L3(0.123)
           ✓ VALID

Layer 9: Valid window = [ 1,  8]
           Top 5 attended: L7(0.134), L1(0.129), L6(0.128), L4(0.128), L2(0.126)
           ✓ VALID

Layer 14: Valid window = [ 6, 13]
           Top 5 attended: L9(0.159), L10(0.128), L11(0.128), L8(0.125), L12(0.124)
           ✓ VALID

Layer 16: Valid window = [ 8, 15]
           Top 5 attended: L8(0.145), L14(0.134), L12(0.132), L13(0.128), L11(0.121)
           ✓ VALID

Layer 18: Valid window = [10, 17]
           Top 5 attended: L16(0.145), L11(0.131), L15(0.123), L10(0.123), L17(0.122)
           ✓ VALID

Layer 20: Valid 

## What Changed?

The lookback constraint is now **enforced at the aggregation level**. This means:

### ✓ Before (INCORRECT)
- Layer 16 could report attending to Layer 1, 2, 3 (violates lookback=8!)
- Layer 18 could report attending to Layer 0, 1 (violates lookback=8!)
- Layer 23 would show early layers in top attended list

### ✓ After (CORRECT)
- Layer 5 (window [0,4]): Only shows layers 0-4 ✓
- Layer 8 (window [0,7]): Only shows layers 0-7 ✓
- Layer 9 (window [1,8]): Only shows layers 1-8 ✓
- Layer 16 (window [8,15]): No spurious early layers ✓
- Layer 23 (window [15,22]): No spurious early layers ✓

**Key insight**: With `lookback=8`, each layer can ONLY attend to the 8 most recent layers. The constraint is now enforced everywhere!

In [11]:
# Show which sources are most attended across the model
print("\n" + "="*80)
print("MOST ATTENDED SOURCES PER ADAPTER LAYER (Aggregate)")
print("="*80)

for layer_idx in range(num_layers):
    top_layers = combined_analyzer.get_most_attended_layers(layer_idx, top_k=5)
    print(f"\nAdapter Layer {layer_idx} attends to:")
    for source_idx, weight in top_layers:
        source_name = "Embedding" if source_idx == -1 else f"Layer {source_idx}"
        print(f"  {source_name}: {weight:.4f}")


MOST ATTENDED SOURCES PER ADAPTER LAYER (Aggregate)

Adapter Layer 0 attends to:
  Embedding: 1.0000

Adapter Layer 1 attends to:
  Embedding: 0.5214
  Layer 0: 0.4786

Adapter Layer 2 attends to:
  Embedding: 0.3470
  Layer 0: 0.3322
  Layer 1: 0.3208

Adapter Layer 3 attends to:
  Layer 0: 0.2882
  Layer 1: 0.2474
  Embedding: 0.2467
  Layer 2: 0.2177

Adapter Layer 4 attends to:
  Layer 2: 0.2394
  Layer 1: 0.2227
  Layer 0: 0.1840
  Layer 3: 0.1774
  Embedding: 0.1766

Adapter Layer 5 attends to:
  Layer 3: 0.1948
  Layer 0: 0.1855
  Embedding: 0.1738
  Layer 4: 0.1637
  Layer 2: 0.1443

Adapter Layer 6 attends to:
  Layer 2: 0.1771
  Layer 1: 0.1617
  Layer 5: 0.1582
  Layer 0: 0.1484
  Layer 3: 0.1368

Adapter Layer 7 attends to:
  Embedding: 0.1804
  Layer 1: 0.1540
  Layer 2: 0.1228
  Layer 3: 0.1211
  Layer 0: 0.1098

Adapter Layer 8 attends to:
  Layer 2: 0.1361
  Layer 5: 0.1326
  Layer 7: 0.1288
  Layer 4: 0.1269
  Layer 3: 0.1233

Adapter Layer 9 attends to:
  Layer 7: 0.

In [12]:
# Focused verification for mid/late layers
print("\n" + "=" * 80)
print("FOCUSED CHECK: lookback window mapping for layers 14-18")
print("=" * 80)

for layer_idx in [0,14, 15, 16, 17, 18]:
    top_sources = combined_analyzer.get_most_attended_layers(layer_idx, top_k=8)
    state_start = max(0, layer_idx + 1 - lookback_window)
    state_end = layer_idx  # state index includes embedding at 0, then layer outputs 1..layer_idx

    # Convert expected state range to expected source ids: -1 for embedding, else layer_idx = state-1
    expected_sources = [(-1 if s == 0 else s - 1) for s in range(state_start, state_end + 1)]

    pretty_expected = ["Embedding" if s == -1 else f"Layer {s}" for s in expected_sources]
    pretty_top = [f"{'Embedding' if s == -1 else f'Layer {s}'}({w:.3f})" for s, w in top_sources]

    print(f"\nAdapter Layer {layer_idx}")
    print(f"  Expected candidate sources: {pretty_expected}")
    print(f"  Top attended sources:      {pretty_top}")

    top_ids = [s for s, _ in top_sources]
    all_valid = all(s in expected_sources for s in top_ids)
    print(f"  Constraint check: {'PASS' if all_valid else 'FAIL'}")


FOCUSED CHECK: lookback window mapping for layers 14-18

Adapter Layer 0
  Expected candidate sources: ['Embedding']
  Top attended sources:      ['Embedding(1.000)']
  Constraint check: PASS

Adapter Layer 14
  Expected candidate sources: ['Layer 6', 'Layer 7', 'Layer 8', 'Layer 9', 'Layer 10', 'Layer 11', 'Layer 12', 'Layer 13']
  Top attended sources:      ['Layer 9(0.159)', 'Layer 10(0.128)', 'Layer 11(0.128)', 'Layer 8(0.125)', 'Layer 12(0.124)', 'Layer 13(0.122)', 'Layer 6(0.117)', 'Layer 7(0.098)']
  Constraint check: PASS

Adapter Layer 15
  Expected candidate sources: ['Layer 7', 'Layer 8', 'Layer 9', 'Layer 10', 'Layer 11', 'Layer 12', 'Layer 13', 'Layer 14']
  Top attended sources:      ['Layer 11(0.148)', 'Layer 7(0.132)', 'Layer 10(0.130)', 'Layer 9(0.124)', 'Layer 14(0.121)', 'Layer 8(0.121)', 'Layer 12(0.121)', 'Layer 13(0.102)']
  Constraint check: PASS

Adapter Layer 16
  Expected candidate sources: ['Layer 8', 'Layer 9', 'Layer 10', 'Layer 11', 'Layer 12', 'Layer 13'

## Step 6: Cross-Dataset Comparison

Compare attention patterns across different tasks.

In [13]:
# Compare task-specific patterns
print("\n" + "="*80)
print("TASK-SPECIFIC ATTENTION PATTERNS")
print("="*80)

dataset_comparison = stats['dataset_comparison']

for dataset_id, dataset_stats in dataset_comparison.items():
    dataset_name = dataset_stats['dataset_name']
    num_samples = dataset_stats['num_samples']
    
    print(f"\n{dataset_name} ({num_samples} samples):")
    print("-" * 60)
    
    for layer_idx in range(min(8, num_layers)):
        if layer_idx in dataset_stats['layer_analysis']:
            layer_info = dataset_stats['layer_analysis'][layer_idx]
            ranking = layer_info.get('layer_ranking', [])[:3]
            mean_gate = layer_info.get('mean_gate', None)
            
            print(f"  Adapter Layer {layer_idx}:")
            if ranking:
                for source_idx, weight in ranking:
                    source_name = "Embedding" if source_idx == -1 else f"Layer {source_idx}"
                    print(f"    ↖ {source_name}: {weight:.4f}")
            if mean_gate is not None:
                print(f"    Gate: {mean_gate:.6f}")


TASK-SPECIFIC ATTENTION PATTERNS

Reasoning (420 samples):
------------------------------------------------------------
  Adapter Layer 0:
    ↖ Embedding: 1.0000
    Gate: 0.100000
  Adapter Layer 1:
    ↖ Embedding: 0.5396
    ↖ Layer 0: 0.4604
    Gate: 0.100000
  Adapter Layer 2:
    ↖ Embedding: 0.3507
    ↖ Layer 0: 0.3473
    ↖ Layer 1: 0.3020
    Gate: 0.100000
  Adapter Layer 3:
    ↖ Layer 0: 0.2924
    ↖ Layer 1: 0.2615
    ↖ Embedding: 0.2316
    Gate: 0.100000
  Adapter Layer 4:
    ↖ Layer 2: 0.2375
    ↖ Layer 1: 0.2184
    ↖ Embedding: 0.1925
    Gate: 0.100000
  Adapter Layer 5:
    ↖ Layer 0: 0.1993
    ↖ Layer 3: 0.1925
    ↖ Layer 4: 0.1652
    Gate: 0.100000
  Adapter Layer 6:
    ↖ Layer 5: 0.1660
    ↖ Layer 2: 0.1626
    ↖ Layer 1: 0.1621
    Gate: 0.100000
  Adapter Layer 7:
    ↖ Embedding: 0.1863
    ↖ Layer 1: 0.1530
    ↖ Layer 2: 0.1228
    Gate: 0.100000

Summarization (420 samples):
------------------------------------------------------------
  Adapter 

## Step 7: Visualizations

Generate visualizations of attention patterns.

In [31]:
# Set up visualization output
output_dir = Path("./attnres_analysis")
output_dir.mkdir(parents=True, exist_ok=True)

visualizer = AttnResVisualizer(output_dir=str(output_dir), dpi=100)
print(f"Visualizations will be saved to: {output_dir}")

Visualizations will be saved to: attnres_analysis


In [32]:
# Generate all visualizations
visualizer.generate_all_plots(stats, num_layers)

print("\n✓ All visualizations generated!")


Generating visualizations...
Output directory: attnres_analysis
Saved: layer_depth_attention_heatmap.png
Saved: gate_activation_across_layers.png
Saved: attention_flow_matrix.png
Saved: layer_ranking_comparison.png
Saved: all_relation_map.png
Saved: cross_dataset_comparison.png
Saved: dataset_gate_distributions.png

All visualizations saved!

✓ All visualizations generated!


## Step 8: Save Analysis Results

In [16]:
# Save analysis to JSON
combined_analyzer.save_analysis(str(output_dir / "attnres_analysis.json"))

print(f"\n✓ Analysis saved to {output_dir / 'attnres_analysis.json'}")

Analysis saved to attnres_analysis/attnres_analysis.json

✓ Analysis saved to attnres_analysis/attnres_analysis.json


## Step 9: Key Insights Summary

In [17]:
# Extract key insights
print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

# 1. Most frequently attended layers
print("\n1. LAYER ATTENDANCE FREQUENCY:")
all_rankings = []
layer_stats = stats['layer_depth_stats']
for layer_idx, layer_stat in layer_stats.items():
    if layer_stat.get('aggregated'):
        ranking = layer_stat['aggregated'].get('layer_ranking', [])
        all_rankings.extend(ranking)

layer_freq = {}
for layer_idx, weight in all_rankings:
    layer_freq[layer_idx] = layer_freq.get(layer_idx, 0) + 1

sorted_layers = sorted(layer_freq.items(), key=lambda x: x[1], reverse=True)
for layer_idx, freq in sorted_layers[:5]:
    print(f"   Layer {layer_idx}: attended by {freq} depth positions")

# 2. Task-specific patterns
print("\n2. TASK-SPECIFIC PATTERNS:")
for dataset_id, dataset_stats in dataset_comparison.items():
    dataset_name = dataset_stats['dataset_name']
    # Find first layer's top attended layer
    if 0 in dataset_stats['layer_analysis']:
        layer_0_info = dataset_stats['layer_analysis'][0]
        ranking = layer_0_info.get('layer_ranking', [])
        if ranking:
            top_source = ranking[0]
            print(f"   {dataset_name}: Early layers attend to layer {top_source[0]} (weight: {top_source[1]:.4f})")

# 3. Gate activation patterns
print("\n3. GATE ACTIVATION PATTERNS:")
gate_stats = stats['gate_stats']
avg_gates = []
for layer_idx in range(min(4, num_layers)):
    gate_stat = gate_stats[layer_idx]
    if gate_stat.get('aggregated'):
        mean_gate = gate_stat['aggregated'].get('mean', 0)
        avg_gates.append(mean_gate)
        print(f"   Layer {layer_idx}: {mean_gate:.6f}")

print(f"\n   Average gate activation (first 4 layers): {np.mean(avg_gates):.6f}")
if np.mean(avg_gates) < 0.01:
    print("   → Gates are learning to activate (moving away from init value)")
else:
    print("   → Adapters are actively contributing to model output")


KEY INSIGHTS

1. LAYER ATTENDANCE FREQUENCY:
   Layer -1: attended by 8 depth positions
   Layer 0: attended by 8 depth positions
   Layer 1: attended by 8 depth positions
   Layer 2: attended by 8 depth positions
   Layer 3: attended by 8 depth positions

2. TASK-SPECIFIC PATTERNS:
   Reasoning: Early layers attend to layer -1 (weight: 1.0000)
   Summarization: Early layers attend to layer -1 (weight: 1.0000)
   Q&A: Early layers attend to layer -1 (weight: 1.0000)

3. GATE ACTIVATION PATTERNS:
   Layer 0: 0.100000
   Layer 1: 0.100000
   Layer 2: 0.100000
   Layer 3: 0.100000

   Average gate activation (first 4 layers): 0.100000
   → Adapters are actively contributing to model output
